In [1]:
import pandas as pd
import numpy as np

print("CashFlowIQ - Data Quality Check")
print("Python environment is working!")

CashFlowIQ - Data Quality Check
Python environment is working!


In [3]:
from pathlib import Path

# Find the CashFlowIQ project folder
project_path = Path.cwd().parent

# Locate the Data folder
data_path = project_path / "Data"

print("Project path:", project_path)
print("Data path:", data_path)
print("Data folder exists:", data_path.exists())

Project path: d:\DATA ANALYST PROJECTS\Analytics_Projects\CashFlowIQ
Data path: d:\DATA ANALYST PROJECTS\Analytics_Projects\CashFlowIQ\Data
Data folder exists: True


In [4]:
customers = pd.read_csv(data_path / "customers.csv")

customers.head()

,customer_id,customer_name,customer_segment,region,credit_terms_days
0,C1001,Customer 001,SMB,East,60
1,C1002,Customer 002,Mid-Market,East,60
2,C1003,Customer 003,Mid-Market,West,60
3,C1004,Customer 004,SMB,West,30
4,C1005,Customer 005,Enterprise,West,15


In [5]:
customers.shape

(120, 5)

In [6]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        120 non-null    object
 1   customer_name      120 non-null    object
 2   customer_segment   120 non-null    object
 3   region             120 non-null    object
 4   credit_terms_days  120 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 4.8+ KB


In [7]:
customers.isnull().sum()

customer_id          0
customer_name        0
customer_segment     0
region               0
credit_terms_days    0
dtype: int64

In [8]:
# List of all tables in our project

table_names = [
    "customers",
    "vendors",
    "products",
    "sales_invoices",
    "vendor_invoices",
    "operating_expenses",
    "payroll",
    "loans",
    "loan_payments",
    "bank_transactions",
    "inventory_snapshots"
]

# Store dataframes in a dictionary
tables = {}

for table in table_names:
    file_path = data_path / f"{table}.csv"
    tables[table] = pd.read_csv(file_path)

print("Tables loaded successfully:")
for table in tables:
    print(table)

Tables loaded successfully:
customers
vendors
products
sales_invoices
vendor_invoices
operating_expenses
payroll
loans
loan_payments
bank_transactions
inventory_snapshots


In [9]:
quality_report = []

for table_name, df in tables.items():

    quality_report.append({
        "Table": table_name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Cells": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum()
    })

quality_report = pd.DataFrame(quality_report)

quality_report

,Table,Rows,Columns,Missing Cells,Duplicate Rows
0,customers,120,5,0,0
1,vendors,45,4,0,0
2,products,60,5,0,0
3,sales_invoices,5000,12,497,0
4,vendor_invoices,2200,7,225,0
5,operating_expenses,160,4,0,0
6,payroll,20,5,0,0
7,loans,3,6,0,0
8,loan_payments,41,6,0,0
9,bank_transactions,6700,5,0,0


In [10]:
for table_name, df in tables.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) > 0:
        print("\n" + "=" * 50)
        print(table_name)
        print("=" * 50)
        print(missing)


sales_invoices
payment_date    497
dtype: int64

vendor_invoices
payment_date    225
dtype: int64


In [11]:
# Primary key columns for each table

primary_keys = {
    "customers": "customer_id",
    "vendors": "vendor_id",
    "products": "product_id",
    "sales_invoices": "invoice_id",
    "vendor_invoices": "vendor_invoice_id",
    "operating_expenses": "expense_id",
    "payroll": "payroll_id",
    "loans": "loan_id",
    "loan_payments": "payment_id",
    "bank_transactions": "bank_txn_id",
    "inventory_snapshots": "inventory_snapshot_id"
}

# Check uniqueness of every primary key

for table_name, key in primary_keys.items():

    df = tables[table_name]

    duplicate_count = df[key].duplicated().sum()
    missing_count = df[key].isnull().sum()

    print(f"{table_name}:")
    print(f"  Primary Key: {key}")
    print(f"  Duplicate IDs: {duplicate_count}")
    print(f"  Missing IDs: {missing_count}")
    print()

customers:
  Primary Key: customer_id
  Duplicate IDs: 0
  Missing IDs: 0

vendors:
  Primary Key: vendor_id
  Duplicate IDs: 0
  Missing IDs: 0

products:
  Primary Key: product_id
  Duplicate IDs: 0
  Missing IDs: 0

sales_invoices:
  Primary Key: invoice_id
  Duplicate IDs: 0
  Missing IDs: 0

vendor_invoices:
  Primary Key: vendor_invoice_id
  Duplicate IDs: 0
  Missing IDs: 0

operating_expenses:
  Primary Key: expense_id
  Duplicate IDs: 0
  Missing IDs: 0

payroll:
  Primary Key: payroll_id
  Duplicate IDs: 0
  Missing IDs: 0

loans:
  Primary Key: loan_id
  Duplicate IDs: 0
  Missing IDs: 0

loan_payments:
  Primary Key: payment_id
  Duplicate IDs: 0
  Missing IDs: 0

bank_transactions:
  Primary Key: bank_txn_id
  Duplicate IDs: 0
  Missing IDs: 0

inventory_snapshots:
  Primary Key: inventory_snapshot_id
  Duplicate IDs: 0
  Missing IDs: 0



In [12]:
# Foreign key relationships to validate

foreign_keys = {
    "sales_invoices": {
        "customer_id": ("customers", "customer_id"),
        "product_id": ("products", "product_id")
    },
    "vendor_invoices": {
        "vendor_id": ("vendors", "vendor_id")
    },
    "inventory_snapshots": {
        "product_id": ("products", "product_id")
    },
    "loan_payments": {
        "loan_id": ("loans", "loan_id")
    }
}

for child_table, relationships in foreign_keys.items():

    print("\n" + "=" * 60)
    print(child_table.upper())
    print("=" * 60)

    child_df = tables[child_table]

    for child_key, (parent_table, parent_key) in relationships.items():

        parent_df = tables[parent_table]

        invalid_values = child_df.loc[
            ~child_df[child_key].isin(parent_df[parent_key]),
            child_key
        ].dropna().unique()

        print(f"\n{child_key} → {parent_table}.{parent_key}")
        print(f"Invalid references: {len(invalid_values)}")

        if len(invalid_values) > 0:
            print("Examples:", invalid_values[:10])


SALES_INVOICES

customer_id → customers.customer_id
Invalid references: 0

product_id → products.product_id
Invalid references: 0

VENDOR_INVOICES

vendor_id → vendors.vendor_id
Invalid references: 0

INVENTORY_SNAPSHOTS

product_id → products.product_id
Invalid references: 0

LOAN_PAYMENTS

loan_id → loans.loan_id
Invalid references: 0


In [13]:
# ============================================================
# DATE & FINANCIAL VALIDATION
# ============================================================

sales = tables["sales_invoices"].copy()
vendors = tables["vendor_invoices"].copy()
expenses = tables["operating_expenses"].copy()
payroll = tables["payroll"].copy()
loans = tables["loans"].copy()
loan_payments = tables["loan_payments"].copy()
bank = tables["bank_transactions"].copy()
inventory = tables["inventory_snapshots"].copy()

print("SALES INVOICE VALIDATION")
print("=" * 50)

# Invoice date should not be after due date
invalid_due_dates = (sales["due_date"] < sales["invoice_date"]).sum()

# Paid invoice should not have payment before invoice
paid_sales = sales[sales["payment_date"].notna()]
invalid_payment_dates = (
    paid_sales["payment_date"] < paid_sales["invoice_date"]
).sum()

# Net amount should equal gross - discount
invalid_invoice_amounts = (
    sales["net_invoice_amount"].round(2)
    != (sales["gross_amount"] - sales["discount_amount"]).round(2)
).sum()

# Negative or zero invoice amounts
invalid_sales_amounts = (sales["net_invoice_amount"] <= 0).sum()

print("Due date before invoice date:", invalid_due_dates)
print("Payment before invoice date:", invalid_payment_dates)
print("Incorrect net invoice amount:", invalid_invoice_amounts)
print("Zero/negative invoice amounts:", invalid_sales_amounts)


print("\nVENDOR INVOICE VALIDATION")
print("=" * 50)

invalid_vendor_due_dates = (
    vendors["due_date"] < vendors["invoice_date"]
).sum()

paid_vendors = vendors[vendors["payment_date"].notna()]

invalid_vendor_payment_dates = (
    paid_vendors["payment_date"] < paid_vendors["invoice_date"]
).sum()

invalid_vendor_amounts = (vendors["invoice_amount"] <= 0).sum()

print("Due date before invoice date:", invalid_vendor_due_dates)
print("Payment before invoice date:", invalid_vendor_payment_dates)
print("Zero/negative invoice amounts:", invalid_vendor_amounts)


print("\nSALES QUANTITY VALIDATION")
print("=" * 50)

invalid_quantities = (sales["quantity"] <= 0).sum()

print("Zero/negative quantities:", invalid_quantities)


print("\nINVENTORY VALIDATION")
print("=" * 50)

invalid_inventory_units = (inventory["units_on_hand"] < 0).sum()
invalid_inventory_values = (inventory["inventory_value"] < 0).sum()

print("Negative inventory units:", invalid_inventory_units)
print("Negative inventory value:", invalid_inventory_values)


print("\nLOAN VALIDATION")
print("=" * 50)

invalid_loan_amounts = (loans["principal_amount"] <= 0).sum()
invalid_interest_rates = (loans["interest_rate"] < 0).sum()

print("Zero/negative loan principal:", invalid_loan_amounts)
print("Negative interest rates:", invalid_interest_rates)


print("\nBANK TRANSACTION VALIDATION")
print("=" * 50)

invalid_bank_dates = bank["transaction_date"].isna().sum()

print("Missing transaction dates:", invalid_bank_dates)


print("\nVALIDATION COMPLETE")

SALES INVOICE VALIDATION
Due date before invoice date: 0
Payment before invoice date: 0
Incorrect net invoice amount: 0
Zero/negative invoice amounts: 0

VENDOR INVOICE VALIDATION
Due date before invoice date: 0
Payment before invoice date: 0
Zero/negative invoice amounts: 0

SALES QUANTITY VALIDATION
Zero/negative quantities: 0

INVENTORY VALIDATION
Negative inventory units: 0
Negative inventory value: 0

LOAN VALIDATION


KeyError: 'interest_rate'

In [14]:
loans.columns.tolist()

['loan_id',
 'loan_type',
 'principal_amount',
 'annual_interest_rate',
 'start_date',
 'tenure_months']

In [15]:
loans.head()

,loan_id,loan_type,principal_amount,annual_interest_rate,start_date,tenure_months
0,L001,Term Loan,8000000,0.105,2025-02-01,48
1,L002,Working Capital Loan,5000000,0.115,2025-07-01,36
2,L003,Equipment Loan,3000000,0.095,2026-01-01,36


In [16]:
# ============================================================
# LOAN VALIDATION - CORRECTED
# ============================================================

print("LOAN VALIDATION")
print("=" * 50)

invalid_loan_amounts = (
    loans["principal_amount"] <= 0
).sum()

invalid_interest_rates = (
    loans["annual_interest_rate"] < 0
).sum()

invalid_tenure = (
    loans["tenure_months"] <= 0
).sum()

invalid_start_dates = (
    loans["start_date"].isna()
).sum()

print("Zero/negative loan principal:", invalid_loan_amounts)
print("Negative interest rates:", invalid_interest_rates)
print("Zero/negative tenure:", invalid_tenure)
print("Missing loan start dates:", invalid_start_dates)

LOAN VALIDATION
Zero/negative loan principal: 0
Negative interest rates: 0
Zero/negative tenure: 0
Missing loan start dates: 0


In [17]:
# ============================================================
# BANK & PAYROLL VALIDATION
# ============================================================

print("BANK TRANSACTION VALIDATION")
print("=" * 50)

print(
    "Missing transaction dates:",
    bank["transaction_date"].isna().sum()
)

print(
    "Missing transaction amounts:",
    bank["amount"].isna().sum()
)


print("\nPAYROLL VALIDATION")
print("=" * 50)

print(
    "Negative gross payroll:",
    (payroll["gross_payroll"] < 0).sum()
)

print(
    "Negative net payroll:",
    (payroll["net_payroll"] < 0).sum()
)

print(
    "Missing pay dates:",
    payroll["pay_date"].isna().sum()
)

BANK TRANSACTION VALIDATION
Missing transaction dates: 0
Missing transaction amounts: 0

PAYROLL VALIDATION
Negative gross payroll: 0
Negative net payroll: 0
Missing pay dates: 0


In [18]:
# ============================================================
# DATA TYPE AUDIT
# ============================================================

for table_name, df in tables.items():

    print("\n" + "=" * 60)
    print(table_name.upper())
    print("=" * 60)

    print(df.dtypes)


CUSTOMERS
customer_id          object
customer_name        object
customer_segment     object
region               object
credit_terms_days     int64
dtype: object

VENDORS
vendor_id             object
vendor_name           object
vendor_category       object
payment_terms_days     int64
dtype: object

PRODUCTS
product_id        object
product_name      object
category          object
unit_cost        float64
selling_price    float64
dtype: object

SALES_INVOICES
invoice_id             object
invoice_date           object
customer_id            object
product_id             object
quantity                int64
discount_pct          float64
unit_price            float64
gross_amount          float64
discount_amount       float64
net_invoice_amount    float64
due_date               object
payment_date           object
dtype: object

VENDOR_INVOICES
vendor_invoice_id     object
invoice_date          object
vendor_id             object
expense_category      object
invoice_amount       flo